In [12]:
import itertools
import re

class LogicSolver:
    def __init__(self):
        # Simplified mapping for standard slide terms
        self.operator_map = {
            'negation': ' not ',
            'and': ' and ',
            'or': ' or '
        }
        self.keywords = {'and', 'or', 'not', 'negation'}

    def clean_formula(self, formula: str) -> str:
        """Translates logic terms to Python operators."""
        cleaned = formula
        for op, py_op in self.operator_map.items():
            cleaned = cleaned.replace(op, py_op)
        return cleaned

    def get_symbols(self, formula) -> set:
        """Extracts unique proposition variables."""
        if isinstance(formula, list):
            symbols = set()
            for f in formula:
                symbols.update(self.get_symbols(f))
            return symbols

        cleaned = self.clean_formula(formula)
        tokens = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', cleaned)
        return {t for t in tokens if t.lower() not in self.keywords}

    def evaluate(self, formula: str, model: dict) -> bool:
        """Evaluates a logic formula string to a boolean."""
        eval_str = self.clean_formula(formula)
        sorted_symbols = sorted(model.keys(), key=len, reverse=True)
        for sym in sorted_symbols:
            eval_str = re.sub(r'\b' + re.escape(sym) + r'\b', str(model[sym]), eval_str)
        return bool(eval(eval_str))

    def run_slide_example(self):
        """Executes the specific slide 5-10 problem and displays a detailed truth table."""
        # 1. Define expressions matching slides 5-10
        # KB components evaluated separately to construct the truth table columns
        kb_sentence1 = "A or C"
        kb_sentence2 = "B or negation(C)"
        alpha = "A or B"

        # Comprehensive evaluation tracking list
        kb = [kb_sentence1, kb_sentence2]
        symbols = sorted(list(self.get_symbols(kb) | self.get_symbols(alpha)))

        # 2. Print Truth Table Header
        print("=" * 70)
        print("  PROPOSITIONAL INFERENCE TRUTH TABLE: ENUMERATION METHOD")
        print("=" * 70)
        print(f"{'A':<8}{'B':<8}{'C':<8}|  {'A ∨ C':<10}{'B ∨ ¬C':<10}|  {'KB':<8}{'α':<8}")
        print("-" * 70)

        # 3. Enumerate all truth models (True first to match standard slide layouts)
        combinations = list(itertools.product([True, False], repeat=len(symbols)))
        all_entailed = True

        for combo in combinations:
            model = dict(zip(symbols, combo))

            # Compute truth column values
            val_c1 = self.evaluate(kb_sentence1, model)
            val_c2 = self.evaluate(kb_sentence2, model)
            val_kb = val_c1 and val_c2
            val_alpha = self.evaluate(alpha, model)

            # Flag if a counterexample is discovered
            status_flag = ""
            if val_kb and not val_alpha:
                status_flag = " ❌ (Counterexample)"
                all_entailed = False
            elif val_kb:
                status_flag = " ⭐️ (Model satisfies KB)"

            # Format row strings
            print(f"{str(model['A']):<8}{str(model['B']):<8}{str(model['C']):<8}|  "
                  f"{str(val_c1):<10}{str(val_c2):<10}|  "
                  f"{str(val_kb):<8}{str(val_alpha):<8}{status_flag}")

        # 4. Print final compilation conclusion
        print("=" * 70)
        print("FINAL RESULT:")
        if all_entailed:
            print("  ✅ Verification Successful: KB ⊨ α (KB entails α).")
            print("  No model exists where KB is True and α is False.")
        else:
            print("  ❌ Verification Failed: KB does not entail α.")
        print("=" * 70)

if __name__ == "__main__":
    solver = LogicSolver()
    solver.run_slide_example()


  PROPOSITIONAL INFERENCE TRUTH TABLE: ENUMERATION METHOD
A       B       C       |  A ∨ C     B ∨ ¬C    |  KB      α       
----------------------------------------------------------------------
True    True    True    |  True      True      |  True    True     ⭐️ (Model satisfies KB)
True    True    False   |  True      True      |  True    True     ⭐️ (Model satisfies KB)
True    False   True    |  True      False     |  False   True    
True    False   False   |  True      True      |  True    True     ⭐️ (Model satisfies KB)
False   True    True    |  True      True      |  True    True     ⭐️ (Model satisfies KB)
False   True    False   |  False     True      |  False   True    
False   False   True    |  True      False     |  False   False   
False   False   False   |  False     True      |  False   False   
FINAL RESULT:
  ✅ Verification Successful: KB ⊨ α (KB entails α).
  No model exists where KB is True and α is False.
